# Data Generator for Neural Network Training on the rotated surface code

This notebook is for generating training data from stim to be used in a pytorch neural network implementation. It maps stim detector events into a tensor compatible with a PyTorch CNN.

PyTorch CNN expects input structure of [batch,channel,height,width]
The time coordinate of the syndrome history gets mapped to the channel dimension, shots becomes the batch, the height and width are the stim detector coordinates

Refs:
https://github.com/quantumlib/Stim/blob/main/doc/getting_started.ipynb
https://github.com/quantumlib/Stim/wiki/Stim-v1.12-Python-API-Reference
https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html
https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html

In [ ]:
%pip install -r ../requirements.txt

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Using cached iniconfig-2.3.0-py3-none-any.whl.metadata (2.5 kB)
  Using cached pluggy-1.6.0-py3-none-any.whl.metadata (4.8 kB)
   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ---------------------------------------- 2.8/2.8 MB 22.5 MB/s  0:00:00
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 1.7/1.7 MB 16.2 MB/s  0:00:00
Using cached pluggy-1.6.0-py3-none-any.whl (20 kB)
Using cached iniconfig-2.3.0-py3-none-any.whl (7.5 kB)
  Created wheel for sinter: filename=sinter-1.16.0-py3-none-any.whl size=197620 sha256=16091d4a4ad2786e9969fca67dfb032227267c021ec1b24176fa2b29a2138345
  Stored in directory: c:\users\kjell\appdata\local\pip\cache\wheels\21\d7\ee\20c9d539daa5f139b411ebf82465352547c07694101973ae3f
Successfully built sinter

   ---------------------------------------- 0/8 [tqdm]
   -----------

  DEPRECATION: Building 'sinter' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'sinter'. Discussion can be found at https://github.com/pypa/pip/issues/6334

[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
""" IMPORTS """
from pathlib import Path
import json

import numpy as np
import matplotlib.pyplot as plt
import stim

## Definitions
First we define our distance, physical error rate, number of shots, and our paths for saving data to.
The label y of the dataset indicates whether a logical observable flipped on the rotated_memory_z surface code.

In [33]:
d = 7
r = 3*d
p = 1e-4
shots = 100_000

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)
dataset_name = f"rotated_surface_code_distance_{d}"

dataset_path = data_dir / f"{dataset_name}.npz"
metadata_path = data_dir / f"{dataset_name}_metadata.json"

## Generate circuit
Generate our rotated surface code.
We apply rotated_memory_z to predict z logical observable flips here. 
Could also switch and train on rotated_memory_x.
Stim also has color_code:memory_xyz, which could be used if we switched to the color code.

In [34]:
circuit = stim.Circuit.generated(
    "surface_code:rotated_memory_z",
    rounds=r,
    distance=d,
    before_round_data_depolarization=p, 
    after_clifford_depolarization=p, 
    before_measure_flip_probability=p, 
    after_reset_flip_probability=p
)

## Sampling
Next we sample detector events and output to a flat np array.

Stim gives us 
- dets: detector events
- obs: observable flips

In [ ]:
sampler = circuit.compile_detector_sampler()

dets, obs = sampler.sample(
    shots=shots,
    separate_observables=True,
)

X_flat = dets.astype(np.float32)
labels = obs[:, 0].astype(np.float32)

print("X shape:", X_flat.shape)
print("labels shape:", labels.shape)
print("logical error rate:", labels.mean())

X shape: (100000, 1008)
y shape: (100000,)
logical error rate: 0.03207



## Map detector samples to CNN grid
We have a flat detector vector now. We need to map this to a rotated surface code tensor compatible with a CNN

Pytorch expects [batch, channels, height, width]

In [ ]:
detector_coords = circuit.get_detector_coordinates()
xs = []
ys = []
ts = []

# Each detector has coordinates [x, y, t]
for coord in detector_coords.values():
    xs.append(coord[0])
    ys.append(coord[1])
    ts.append(coord[2])

# Grid only needs unique coordinates, so get rid of duplicates and sort
unique_xs = sorted(set(xs))
unique_ys = sorted(set(ys))
unique_ts = sorted(set(ts))

# Physical coordinates to array indices
x_to_i = {x: i for i, x in enumerate(unique_xs)}
y_to_i = {y: i for i, y in enumerate(unique_ys)}
t_to_i = {t: i for i, t in enumerate(unique_ts)}

height = len(unique_ys)
width = len(unique_xs)
time_slices = len(unique_ts)

# Dictionary mapping detector to index in the CNN input array
detector_to_grid = {}

for det_idx, coord in detector_coords.items():
    x = coord[0]
    y = coord[1]
    t = coord[2] if len(coord) > 2 else 0

    # [channel, row, column] = [t, y, x]
    detector_to_grid[int(det_idx)] = (
        t_to_i[t],
        y_to_i[y],
        x_to_i[x],
    )

print("time slices:", time_slices)
print("height:", height)
print("width:", width)
print("CNN input shape:", (time_slices, height, width))

time slices: 22
height: 8
width: 8
CNN input shape: (22, 8, 8)


## Build CNN Tensor

In [37]:
num_shots, num_detectors = X_flat.shape

# Create array for CNN input and map detector measurements to indices
X_cnn = np.zeros(
    (num_shots, time_slices, height, width),
    dtype=np.float32,
)

for det_idx in range(num_detectors):
    t_i, y_i, x_i = detector_to_grid[det_idx]
    X_cnn[:, t_i, y_i, x_i] = X_flat[:, det_idx]

print("CNN X shape:", X_cnn.shape)
print("labels shape:", labels.shape)

CNN X shape: (100000, 22, 8, 8)
labels shape: (100000,)


In [ ]:
# Save the dataset and metadata
metadata = {
    "dataset_name": dataset_name,
    "task": "surface_code:rotated_memory_z",
    "distance": d,
    "rounds": r,
    "physical_error_rate": p,
    "shots": shots,

    "num_detectors": int(circuit.num_detectors),
    "num_observables": int(circuit.num_observables),
    
    "input_shape": list(X_cnn.shape[1:]),
    "time_slices": time_slices,
    "height": height,
    "width": width,

    "unique_x_coords": xs,
    "unique_y_coords": ys,
    "unique_t_coords": ts,
}

np.savez_compressed(
    dataset_path,
    X=X_cnn,
    y=labels,
    X_flat=X_flat,
)

with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=4)

print("Saved CNN dataset:", dataset_path)
print("Saved metadata:", metadata_path)

Saved CNN dataset: data\rotated_surface_code_distance_7.npz
Saved metadata: data\rotated_surface_code_distance_7_metadata.json


## Remarks
Generated tensor is now prepared in the shape
[shots,time_slices,height,width]